<h1>2.3 编译执行：从 ROOT macro 到独立程序</h1>
<h2>1. 运行方式</h2><p>2.2 中的宏由 ROOT 的 Cling 即时编译。<code>.L tracking.C+</code> 则使用 ACLiC 编译并加载宏。本节把同一算法编译为终端中可调用的程序，方便传入 run 号、批量处理文件。改变的是程序入口和文件管理，不是 tracking 方法。</p>
<h2>2. 目录与 MakeClass</h2><p>本例位于 <code>code/compile1</code>。<code>main.cpp</code> 管理输入输出，<code>include/tracking.h</code> 声明类，<code>src/tracking.C</code> 保存 2.2 的计算过程。生成 MakeClass 框架时不要覆盖已经修改的分析文件；可以在新目录中生成并比较。</p>

<h2>2. 主程序：参数、文件、分析调用</h2><p><code>argc</code> 是参数个数，<code>argv[1]</code> 是 run 号。没有提供目录时，输入在章目录，输出在本例程序目录。下面代码与实际 main.cpp 一致；读取失败时返回非零值，供批处理脚本判断。</p><pre><code class="language-cpp">#include &lt;TFile.h&gt;
#include &lt;TTree.h&gt;
#include &lt;TString.h&gt;
#include &lt;cstdlib&gt;
#include &lt;iostream&gt;
#include "tracking.h"

int main(int argc, char** argv) {
    if (argc!=2 &amp;&amp; argc!=4) {
        std::cerr &lt;&lt; "Usage: ./tracking run [input_dir output_dir]\n";
        return 1;
    }
    char* end = nullptr;
    long parsed = std::strtol(argv[1], &amp;end, 10);
    if (end==argv[1] || *end!='\0' || parsed&lt;0 || parsed&gt;999999) {
        std::cerr &lt;&lt; "Invalid run number: " &lt;&lt; argv[1] &lt;&lt; '\n';
        return 1;
    }
    int run = int(parsed);
    const char* inputDir = argc==4 ? argv[2] : "../..";
    const char* outputDir = argc==4 ? argv[3] : ".";
    TString inputName = Form("%s/f8ppac%03d.root",inputDir,run);
    TString outputName = Form("%s/out%03d.root",outputDir,run);
    TFile* input = TFile::Open(inputName);
    if (!input || input-&gt;IsZombie()) {
        std::cerr &lt;&lt; "Cannot open " &lt;&lt; inputName &lt;&lt; '\n';
        delete input;
        return 1;
    }
    TTree* tin = input-&gt;Get&lt;TTree&gt;("tree");
    if (!tin) {
        std::cerr &lt;&lt; "Missing tree in " &lt;&lt; inputName &lt;&lt; '\n';
        delete input;
        return 1;
    }
    TFile output(outputName,"RECREATE");
    if (output.IsZombie()) return 1;
    TTree* tout = new TTree("tree","PPAC tracking");
    {
        tracking analysis(tin);
        analysis.Loop(tout);

        std::cout &lt;&lt; "Input=" &lt;&lt; tin-&gt;GetEntries() &lt;&lt; ", output=" &lt;&lt; tout-&gt;GetEntries() &lt;&lt; '\n';
        output.Write();
    } // MakeClass 基类析构时释放输入文件；不再重复 delete input。
    return 0;
}</code></pre>

<h2>3. Makefile：编译与链接</h2><pre><code class="language-cpp">CXX = c++
CPPFLAGS = -Iinclude $(shell root-config --cflags)
CXXFLAGS = -O2 -Wall
LDLIBS = $(shell root-config --libs)
SOURCES = main.cpp $(wildcard src/*.cpp src/*.C)
HEADERS = $(wildcard include/*.h)

all: tracking

tracking: $(SOURCES) $(HEADERS)
	$(CXX) $(CPPFLAGS) $(CXXFLAGS) $(SOURCES) $(LDLIBS) -o $@

clean:
	rm -f tracking</code></pre><p><code>root-config --cflags</code> 提供 ROOT 头文件及编译选项，<code>--libs</code> 提供链接库。包含头文件解决声明问题，链接解决函数实现问题，两者不能互相替代。修改源码后重新执行 <code>make</code>；编译失败先看第一条 error。</p>

<h2>4. 运行和核对</h2><p>在章目录执行下面两行。程序读取 <code>f8ppac001.root</code>，写出 <code>code/compile1/out001.root</code>。原始文件不变。使用同一输入与选择条件时，输出事例编号、靶点位置及拟合参数应与 2.2 一致，而不只是比较最终事例数。</p>

In [5]:
!make -C code/compile1
!cd code/compile1 && ./tracking 1

c++ -Iinclude -stdlib=libc++ -pthread -std=c++17 -m64 -I/opt/homebrew/Cellar/root/6.40.02/include/root -O2 -Wall main.cpp src/tracking.C -L/opt/homebrew/Cellar/root/6.40.02/lib/root -lCore -lImt -lRIO -lNet -lHist -lGraf -lGraf3d -lGpad -lROOTVecOps -lTree -lTreePlayer -lRint -lPostscript -lMatrix -lPhysics -lMathCore -lThread -lROOTNTuple -lROOTNTupleUtil -lMultiProc -lROOTDataFrame -stdlib=libc++ -Wl,-rpath,/opt/homebrew/Cellar/root/6.40.02/lib/root -lpthread -lm -ldl -o tracking
Processing Event: 30000 / 739685
Processing Event: 50000 / 739685
Processing Event: 110000 / 739685
Processing Event: 130000 / 739685
Processing Event: 140000 / 739685
Processing Event: 150000 / 739685
Processing Event: 190000 / 739685
Processing Event: 240000 / 739685
Processing Event: 270000 / 739685
Processing Event: 360000 / 739685
Processing Event: 390000 / 739685
Processing Event: 400000 / 739685
Processing Event: 410000 / 739685
Processing Event: 440000 / 739685
Processing Event: 450000 / 739685
Proce

<p>完整源码：<a href="code/compile1/main.cpp">main.cpp</a> · <a href="code/compile1/include/tracking.h">tracking.h</a> · <a href="code/compile1/src/tracking.C">tracking.C</a></p>